# Ntoma — train the Ghanaian fabric classifier

Fine-tunes a MobileNetV3-Small on the Ntoma Ghana fabric taxonomy and exports an
int8 TFLite model that drops into the Android app behind the existing `AnalysisEngine`
interface.

**Before you run — two notebook settings, both required:**

| Setting | Value | Why |
|---|---|---|
| Accelerator | `GPU T4 x2` (or P100) | CPU training is ~50x slower |
| Internet | `On` | needed to fetch ImageNet weights |

**Add your dataset first:** *Add Input* -> your `ntoma-fabric` dataset. It should contain
`raw/<CLASS>/*.jpg` (build it with `tools/dataset/kaggle/make_kaggle_dataset.py`).

### What this notebook does NOT do

It does not make a model accurate. It reports honest per-class accuracy on **held-out
capture sessions**, and if the photos are not there yet the numbers will say so. The
2026-09-01 starter run on 83 web photos scored MACRO 0.419 with KENTE 0.00 — this
pipeline is designed to make that kind of result impossible to miss, not to hide it.


In [ ]:
# 1. Environment
import sys, os, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                     '--format=csv,noheader'],capture_output=True,text=True).stdout or
      'No GPU detected — switch the accelerator to GPU T4 x2 in the right-hand panel.')

import tensorflow as tf
print('tensorflow', tf.__version__)
print('GPUs', [d.name for d in tf.config.list_physical_devices('GPU')] or 'none')
if not tf.config.list_physical_devices('GPU'):
    print('\n!! Training on CPU will take hours instead of minutes.')


In [ ]:
# 2. Get the training script
# Clones the repo that holds train_kaggle.py. If your work lives on another branch,
# change BRANCH below to match.
REPO   = 'https://github.com/Jacinth-19/ntoma-studio.git'
BRANCH = 'main'

import os, subprocess, glob
workdir = '/kaggle/working/repo' if os.path.isdir('/kaggle/working') else './repo'
if not os.path.isdir(workdir):
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,workdir],check=False)

hits = glob.glob(os.path.join(workdir,'**','train_kaggle.py'), recursive=True)
if hits:
    SCRIPT = hits[0]
    print('found', SCRIPT)
else:
    SCRIPT = None
    print('could not find train_kaggle.py.\n'
          'Upload the repo as a Kaggle Dataset and set SCRIPT to its path, e.g.\n'
          "  SCRIPT = '/kaggle/input/ntoma-repo/NtomaStudio/tools/dataset/kaggle/train_kaggle.py'")


In [ ]:
# 3. Locate the dataset (we look for a folder containing raw/<CLASS>/*.jpg)
import glob, os
cands = glob.glob('/kaggle/input/*/raw') + glob.glob('/kaggle/input/*/*/raw')
if not cands:
    cands = [p for p in glob.glob('/kaggle/input/*/*') if os.path.isdir(os.path.join(p,'KENTE_ASHANTI'))]
print('candidates:', cands)
DATA = '/kaggle/input/ntoma-fabric/raw'  # <- edit if the line above found something else
if cands:
    DATA = cands[0]
    classes = [d for d in os.listdir(DATA) if os.path.isdir(os.path.join(DATA,d))]
    n = sum(len([f for f in os.listdir(os.path.join(DATA,c)) if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))]) for c in classes)
    print(f'using {DATA}: {len(classes)} classes, {n:,} photos')
else:
    print('Add your dataset via Add Input, then re-run this cell.')


In [ ]:
# 4. Inspect the split BEFORE spending GPU time on it.
# This is the cell that tells you whether a trustworthy accuracy number is even
# possible yet. Sessions are dealt whole, so a class with <3 sessions cannot be
# validated and is excluded from accuracy claims.
!python3 $SCRIPT --data $DATA --out /kaggle/working --dry-run


In [ ]:
# 5. Train.
# On a T4 with ~50k photos at 224px this is roughly 1-3 hours for both phases.
# Skip this cell and go to 5b for a 3-minute pipeline smoke test first.
!python3 $SCRIPT --data $DATA --out /kaggle/working --size 224 --backbone mobilenetv3small


In [ ]:
# 5b. Optional smoke test — proves the plumbing in ~3 minutes, proves nothing else.
# A model from this cell must never be shipped or quoted.
# !python3 $SCRIPT --data $DATA --out /kaggle/working/smoke --smoke --no-pretrained


In [ ]:
# 6. Results
import json, os
p = '/kaggle/working/fabric_labels.json'
if os.path.exists(p):
    d = json.load(open(p))
    print('classes:', len(d['classes']))
    for split, s in d['per_class'].get('_summary', {}).items():
        print(f"  {split}: MACRO {s['macro']:.3f} | OVERALL {s['overall']:.3f} (n={s['n']})")
    print('\ncritical pairs (these decide shippability):')
    for k, v in d.get('critical_pairs', {}).items():
        print(f"  {k}: {v['pair_accuracy']:.3f}  {v['mutual_confusions']} mutual confusions")
    if d.get('unsplittable_classes'):
        print('\nNO accuracy claim possible for:', d['unsplittable_classes'])
else:
    print('no fabric_labels.json yet — run cell 5')


In [ ]:
# 7. Ship it
# Downloads the two files the app needs. The .tflite is int8, uint8 in / uint8 out —
# the Android side must therefore feed uint8 [1,224,224,3] and dequantise the output
# (scale in the json).
import os, json
for f in ['fabric_ghana.tflite','fabric_labels.json','split_report.json']:
    p = f'/kaggle/working/{f}'
    print(('OK   ' if os.path.exists(p) else 'MISS '), f, os.path.getsize(p) if os.path.exists(p) else '')

try:
    from IPython.display import FileLink, display
    display(FileLink('fabric_ghana.tflite'))
    display(FileLink('fabric_labels.json'))
except Exception:
    print('download from the Output tab on the right')

# Then, in the repo:
#   cp fabric_ghana.tflite  app/src/main/assets/
#   cp fabric_labels.json   app/src/main/assets/
# and keep ON_DEVICE_DEMO labelling for every class that did not clear its bar.
